In [ ]:
# %% [1] 初始化设置
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import TheilSenRegressor
import pymannkendall as mk
import os
import matplotlib as mpl
from alibi.explainers import ALE  # 正确的ALE实现库

# 设置中文显示
mpl.rcParams['font.family'] = 'SimHei'
mpl.rcParams['axes.unicode_minus'] = False

# 基础路径设置
base_output_dir = "your_path_here"
os.makedirs(base_output_dir, exist_ok=True)

# 用户自定义参数
target_col = "NPP"
core_features = ['Precipitation_Total', 'Temperature', 'SW5', 'SW7'] # 只分析SW7作为核心因子
interaction_depth = 5

In [ ]:
# %% [2] 数据准备
# 读取数据
data_path = "your_path_here"
data = pd.read_excel(data_path)

# 提取特征和目标
feature_cols = ['Precipitation_Total', 'Precipitation_Frequency', 'Precipitation_Intensity',
                'Dry_Spell_Length', 'Temperature', 'SW5', 'SW6', 'SW7', 'SW8']
features = data[feature_cols]
target = data[target_col]

print("数据加载完成!")
print(f"特征数: {len(feature_cols)}, 样本数: {len(data)}")

In [ ]:
# %% [3] 模型训练
# 训练随机森林模型
print("开始训练随机森林模型...")
rf = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, max_features=0.3)
rf.fit(features, target)
print(f"模型训练完成! OOB分数: {rf.oob_score_:.4f}")

In [ ]:
# %% [4] SHAP单变量分析-数据计算与导出
# 创建输出目录
single_var_dir = os.path.join(base_output_dir, "单变量分析")
os.makedirs(single_var_dir, exist_ok=True)

# 计算SHAP值
print("计算SHAP值中...")
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(features)

# 存储结果
shap_trend_results = []
shap_dependency_data = pd.DataFrame()  # 用于存储依赖关系数据
shap_dependency_data["Index"] = features.index  # 保留索引用于对齐

print("\n开始单变量数据提取...")
for feature in features.columns:
    # 提取数据
    feature_values = features[feature].values
    feature_shap_values = shap_values[:, features.columns.get_loc(feature)]
    
    # 将当前特征的数据添加到DataFrame
    shap_dependency_data[f"{feature}_原始值"] = feature_values
    shap_dependency_data[f"{feature}_SHAP值"] = feature_shap_values
    
    # Theil-Sen回归
    model = TheilSenRegressor(random_state=42)
    model.fit(feature_values.reshape(-1, 1), feature_shap_values)
    slope = model.coef_[0]
    
    # Mann-Kendall趋势检验
    mk_result = mk.original_test(feature_shap_values)
    
    # 存储趋势结果
    shap_trend_results.append({
        "特征": feature,
        "Theil-Sen斜率": slope,
        "Mann-Kendall_p值": mk_result.p,
        "是否显著(p<0.05)": mk_result.p < 0.05,
        "趋势方向": "上升" if slope > 0 else "下降"
    })
    
    print(f"  完成特征: {feature}")

# 保存单变量分析结果
shap_trend_df = pd.DataFrame(shap_trend_results)
shap_trend_df.to_csv(
    os.path.join(base_output_dir, f"{target_col}_shap_趋势分析.csv"), 
    index=False, 
    encoding='utf-8-sig'
)

# 保存依赖关系原始数据
dependency_file = os.path.join(base_output_dir, f"{target_col}_shap_依赖数据.xlsx")
shap_dependency_data.to_excel(dependency_file, index=False, engine='openpyxl')

print("✅ 单变量数据分析完成!")
print(f"趋势分析保存至: {base_output_dir}/{target_col}_shap_趋势分析.csv")
print(f"依赖数据保存至: {dependency_file}")

In [ ]:
# %% [5] SHAP单变量分析
import matplotlib.pyplot as plt
import numpy as np
import os  # 确保 os 被导入

print("\n开始生成单变量分析图表...")

# 设置全局图表参数
FIG_SIZE = (10, 4)      # 图表尺寸 (宽度, 高度)
POINT_SIZE = 36        # 点的大小 (默认值36)
POINT_COLOR = '#78909C' # 点的颜色 (十六进制或颜色名)
POINT_ALPHA = 0.65     # 点的透明度 (0-1)
TREND_COLOR = '#e74c3c'# 趋势线颜色
TREND_WIDTH = 2.5      # 趋势线粗细
DPI = 150              # 图片分辨率

for feature in features.columns:
    print(f"  生成: {feature}")
    
    # 创建图表
    plt.figure(figsize=FIG_SIZE)
    
    # 获取当前特征的数据
    feature_idx = features.columns.get_loc(feature)
    x = features[feature].values
    y = shap_values[:, feature_idx]
    
    # 绘制散点图
    plt.scatter(
        x, y, 
        s=POINT_SIZE, 
        c=POINT_COLOR, 
        alpha=POINT_ALPHA
    )
    
    # 添加标签和标题
    plt.xlabel(feature, fontsize=22)
    plt.ylabel(f"Shap Value", fontsize=22)
    plt.title(f" ", fontsize=14)
    
    # 添加网格线
    plt.grid(True, alpha=0.2)

    # 显式设置边框样式
    ax = plt.gca()
    for spine in ['top', 'bottom', 'left', 'right']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('black')
        ax.spines[spine].set_linewidth(0.8)

    # 设置主刻度线，只保留左和下
    ax.tick_params(
        axis='x',
        which='major',
        bottom=True,
        top=False,
        direction='out',
        length=3,
        width=0.8,
        colors='black',
        labelsize=16
    )
    ax.tick_params(
        axis='y',
        which='major',
        left=True,
        right=False,
        direction='out',
        length=3,
        width=0.8,
        colors='black',
        labelsize=16
    )
    
    # 保存图表
    save_path = os.path.join(single_var_dir, f"SHAP_{feature}_to_{target_col}.png")
    plt.savefig(save_path, dpi=DPI, bbox_inches='tight')
    plt.close()

print("✅ 图表生成完成!")
print(f"图表保存至: {single_var_dir}")


In [ ]:
# %% [6] ALE分析（核心因子）
import warnings
import numpy as np
import matplotlib.pyplot as plt
from alibi.explainers import ALE
import os

# 创建输出目录
ale_dir = os.path.join(base_output_dir, "ALE分析")
os.makedirs(ale_dir, exist_ok=True)

print("\n开始ALE分析...")

# 创建ALE解释器
ale_explainer = ALE(
    rf.predict, 
    feature_names=features.columns.tolist(),
    low_resolution_threshold=len(features) * 0.9  # 提高分辨率
)

# 图表样式配置
ALE_CONFIG = {
    'figsize': (10, 4),
    'main_color': '#880E4F',      # 主线条颜色
    'zero_color': '#546E7A',      # 零线颜色
    'linewidth': 3,               # 主线条粗细
    'dpi': 150,                   # 图片分辨率
    'title_fontsize': 16,
    'label_fontsize': 22,
    'tick_fontsize': 14,
    'grid_alpha': 0.3,
    'quantile_lines': False       # 是否显示分位数标记
}

# 计算ALE值
print("计算ALE值中...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ale_exp = ale_explainer.explain(features.values)

# 一阶ALE分析
print("\n生成一阶ALE图...")
for feature in core_features:
    print(f"  分析特征: {feature}")
    
    # 获取特征索引
    feat_idx = features.columns.get_loc(feature)
    
    # 创建图形
    fig, ax = plt.subplots(figsize=ALE_CONFIG['figsize'])
    
    # 直接从ALE对象获取数据
    deciles = np.percentile(features[feature], np.linspace(0, 100, 11))
    ale_values = ale_exp.ale_values[feat_idx]
    feature_values = ale_exp.feature_values[feat_idx]
    
    # 绘制ALE曲线
    ax.plot(
        feature_values, 
        ale_values, 
        color=ALE_CONFIG['main_color'], 
        linewidth=ALE_CONFIG['linewidth']
    )

     # 显式设置边框样式
    ax = plt.gca()
    for spine in ['top', 'bottom', 'left', 'right']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('black')
        ax.spines[spine].set_linewidth(0.8)

    # 设置主刻度线，只保留左和下
    ax.tick_params(
        axis='x',
        which='major',
        bottom=True,
        top=False,
        direction='out',
        length=3,
        width=0.8,
        colors='black',
        labelsize=16
    )
    ax.tick_params(
        axis='y',
        which='major',
        left=True,
        right=False,
        direction='out',
        length=3,
        width=0.8,
        colors='black',
        labelsize=16
    )

    # 显式开启主刻度线（避免某些环境下被关闭）
    ax.xaxis.set_tick_params(which='major', bottom=True, top=False)
    ax.yaxis.set_tick_params(which='major', left=True, right=False)

    # 添加零线
    ax.axhline(0, color=ALE_CONFIG['zero_color'], linestyle='--', alpha=0.7)
    
    # 可选：显示分位数标记
    if ALE_CONFIG['quantile_lines']:
        ax.vlines(
            deciles, 
            min(ale_values), 
            max(ale_values), 
            color='gray', 
            linestyle=':', 
            alpha=0.7
        )
    
    # 设置标题和标签（不加粗字体）
    ax.set_title(
        f" ", 
        fontsize=ALE_CONFIG['title_fontsize'],
        fontweight='normal'  # 不加粗
    )
    ax.set_xlabel(feature, fontsize=ALE_CONFIG['label_fontsize'], fontweight='normal')
    ax.set_ylabel("ALE", fontsize=ALE_CONFIG['label_fontsize'], fontweight='normal')
    

    
    # 添加网格
    ax.grid(True, alpha=ALE_CONFIG['grid_alpha'])
    
    # 保存图片
    plt.tight_layout()
    plt.savefig(
        os.path.join(ale_dir, f"ALE_{feature}.png"), 
        dpi=ALE_CONFIG['dpi']
    )
    plt.close(fig)

print("✅ ALE分析完成!")
print(f"结果保存至: {ale_dir}")


In [ ]:
# %% [Bootstrap-1] 初始化设置
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from alibi.explainers import ALE
import os
import warnings

# 基础输出路径（新的bootstrap路径）
base_output_dir = r"your_path_here"

os.makedirs(base_output_dir, exist_ok=True)

# 分析目标
target_col = "NPP"

# 核心特征
core_features = ['Precipitation_Total', 'Temperature', 'SW5', 'SW7']

# Bootstrap参数
n_bootstrap = 100
random_seed = 42

print("✅ Bootstrap初始化完成")

In [ ]:
# %% [Bootstrap-2] 数据读取

# 数据路径
data_path = r"your_path_here"

# 读取数据
data = pd.read_excel(data_path)

# 特征
feature_cols = [
    'Precipitation_Total',
    'Precipitation_Frequency',
    'Precipitation_Intensity',
    'Dry_Spell_Length',
    'Temperature',
    'SW5',
    'SW6',
    'SW7',
    'SW8'
]

features = data[feature_cols]
target = data[target_col]

print("✅ 数据读取完成")
print(f"样本数: {len(data)}")
print(f"特征数: {len(feature_cols)}")

In [ ]:
# %% [Bootstrap-3] Threshold detection based on ALE crossing

def detect_ale_threshold(feature_values, ale_values):
    """
    检测ALE与0线的crossing threshold
    
    规则：
    1. 找所有 crossing
    2. 选择 crossing 后符号长期稳定的点
    3. 优先选择最靠右且稳定的 crossing
    """

    x = np.array(feature_values)
    y = np.array(ale_values)

    signs = np.sign(y)

    thresholds = []

    for i in range(len(signs)-1):

        # crossing
        if signs[i] != signs[i+1]:

            # crossing后的符号
            post_sign = signs[i+1]

            # crossing后区域
            right_section = signs[i+1:]

            # 稳定比例
            stability = np.mean(right_section == post_sign)

            # 至少80%一致
            if stability >= 0.8:

                # 线性插值
                x1, x2 = x[i], x[i+1]
                y1, y2 = y[i], y[i+1]

                threshold = x1 - y1 * (x2 - x1) / (y2 - y1)

                thresholds.append(threshold)

    # 无crossing
    if len(thresholds) == 0:
        return np.nan

    # 返回最靠右 crossing
    return thresholds[-1]

print("✅ Threshold detection function ready")

In [ ]:
# %% Full-data ALE threshold detection

print("\nFull-data ALE threshold detection:\n")

# ALE解释器
ale_explainer_full = ALE(
    rf.predict,
    feature_names=features.columns.tolist(),
    low_resolution_threshold=50
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ale_exp_full = ale_explainer_full.explain(features.values)

# 保存threshold
full_thresholds = {}

for feature in core_features:

    feat_idx = features.columns.get_loc(feature)

    # 🚨 正确提取方式
    ale_values = ale_exp_full.ale_values[feat_idx][:, 0]
    feature_values = ale_exp_full.feature_values[feat_idx]

    # threshold detection
    threshold = detect_ale_threshold(
        feature_values,
        ale_values
    )

    full_thresholds[feature] = threshold

    print(f"{feature}: {threshold:.2f}")

    # 画图检查
    plt.figure(figsize=(8,4))
    plt.plot(feature_values, ale_values)
    plt.axhline(0, linestyle='--')

    if not np.isnan(threshold):
        plt.axvline(
            threshold,
            color='red',
            linestyle='--'
        )

    plt.xlabel(feature)
    plt.ylabel("ALE")
    plt.title(f"{feature} threshold = {threshold:.2f}")

    plt.show()

print("\n✅ Full-data threshold detection completed")

In [ ]:
# %% [Bootstrap-4] 单次ALE threshold测试

# RF模型
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    oob_score=True,
    max_features=0.3
)

rf.fit(features, target)

print(f"OOB: {rf.oob_score_:.4f}")

# ALE解释器
ale_explainer = ALE(
    rf.predict,
    feature_names=features.columns.tolist(),
    low_resolution_threshold=50
)

# 计算ALE
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ale_exp = ale_explainer.explain(features.values)

# threshold测试
for feature in core_features:

    feat_idx = features.columns.get_loc(feature)

    # 🚨 正确提取方式
    ale_values = ale_exp.ale_values[feat_idx][:, 0]
    feature_values = ale_exp.feature_values[feat_idx]

    threshold = detect_ale_threshold(
        feature_values,
        ale_values
    )

    print(f"{feature} threshold: {threshold:.2f}")

    # 可视化检查
    plt.figure(figsize=(8,4))
    plt.plot(feature_values, ale_values)
    plt.axhline(0, linestyle='--')

    if not np.isnan(threshold):
        plt.axvline(threshold, color='red', linestyle='--')

    plt.title(f"{feature}: {threshold:.2f}")
    plt.xlabel(feature)
    plt.ylabel("ALE")

    plt.show()

In [ ]:
# %% [Bootstrap-5] Bootstrap threshold analysis

# 保存结果
bootstrap_results = {
    feature: []
    for feature in core_features
}

print("\n开始Bootstrap分析...")

for b in range(n_bootstrap):

    # Bootstrap重采样
    sample_idx = np.random.choice(
        len(features),
        size=len(features),
        replace=True
    )

    X_boot = features.iloc[sample_idx]
    y_boot = target.iloc[sample_idx]

    # RF模型
    rf_boot = RandomForestRegressor(
        n_estimators=100,
        random_state=b,
        oob_score=False,
        max_features=0.3
    )

    rf_boot.fit(X_boot, y_boot)

    # ALE解释器
    ale_explainer = ALE(
        rf_boot.predict,
        feature_names=features.columns.tolist(),
        low_resolution_threshold=50
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ale_exp = ale_explainer.explain(X_boot.values)

    # threshold提取
    for feature in core_features:

        feat_idx = features.columns.get_loc(feature)

        # 🚨 正确ALE提取
        ale_values = ale_exp.ale_values[feat_idx][:, 0]
        feature_values = ale_exp.feature_values[feat_idx]

        threshold = detect_ale_threshold(
            feature_values,
            ale_values
        )

        bootstrap_results[feature].append(threshold)

    # 进度
    if (b + 1) % 10 == 0:
        print(f"已完成 {b+1}/{n_bootstrap}")

print("\n✅ Bootstrap分析完成")

In [ ]:
# %% [Bootstrap-6] Bootstrap结果统计

summary_results = []

print("\nBootstrap Threshold Summary:\n")

for feature in core_features:

    values = np.array(bootstrap_results[feature])

    # 删除NaN
    values = values[~np.isnan(values)]

    mean_val = np.mean(values)
    std_val = np.std(values)

    ci_lower = np.percentile(values, 2.5)
    ci_upper = np.percentile(values, 97.5)

    summary_results.append({
        "Feature": feature,
        "Mean_Threshold": mean_val,
        "Std": std_val,
        "CI_2.5%": ci_lower,
        "CI_97.5%": ci_upper,
        "N": len(values)
    })

    print(f"{feature}")
    print(f"  Mean Threshold : {mean_val:.2f}")
    print(f"  Std            : {std_val:.2f}")
    print(f"  95% CI         : {ci_lower:.2f} – {ci_upper:.2f}")
    print()

# 保存结果
summary_df = pd.DataFrame(summary_results)

summary_path = os.path.join(
    base_output_dir,
    f"{target_col}_bootstrap_threshold_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding='utf-8-sig'
)

print("✅ Bootstrap统计结果保存完成")
print(summary_path)

In [ ]:
# %% [Bootstrap-7] Bootstrap Threshold Violin Plots

import matplotlib.pyplot as plt
import numpy as np
import os

# 输出路径
violin_dir = os.path.join(base_output_dir, "Bootstrap_ViolinPlots")
os.makedirs(violin_dir, exist_ok=True)

print("\n开始绘制Bootstrap Threshold Violin Plots...")

for feature in core_features:

    values = np.array(bootstrap_results[feature])

    # 删除 NaN
    values = values[~np.isnan(values)]

    # 分位数
    q25 = np.percentile(values, 25)
    q50 = np.percentile(values, 50)
    q75 = np.percentile(values, 75)

    # 创建图
    fig, ax = plt.subplots(figsize=(7, 2.5))

    # Horizontal violin plot
    violin = ax.violinplot(
        values,
        vert=False,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    # 设置 violin 样式
    for body in violin['bodies']:
        body.set_alpha(0.7)

    # Quartile lines
    ax.axvline(
        q25,
        linestyle='--',
        linewidth=1.5,
        label='25% quantile'
    )

    ax.axvline(
        q50,
        linestyle='-',
        linewidth=2,
        label='Median'
    )

    ax.axvline(
        q75,
        linestyle='--',
        linewidth=1.5,
        label='75% quantile'
    )

    # 图例
    ax.legend(
        frameon=False,
        fontsize=10,
        loc='upper right'
    )

    # 标签
    ax.set_xlabel(f"{feature} Threshold", fontsize=13)
    ax.set_yticks([])

    # 标题
    ax.set_title(
        f"Bootstrap Threshold Distribution: {feature}",
        fontsize=14
    )

    # 网格
    ax.grid(alpha=0.25)

    # 边框
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(0.8)

    # 保存
    save_path = os.path.join(
        violin_dir,
        f"{feature}_bootstrap_violin.png"
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"✅ {feature} 完成")

print("\n✅ 所有 Violin Plot 绘制完成")
print(violin_dir)